# Setup (MULTI!)

The following cell holds the definition of our parameters, these values can be overriden by rendering the with e.g. the following command:

papermill -p alpha 0.2 -p ratio 0.3 universe_analysis.ipynb output/test_run.ipynb

In [ ]:
settings = {
    "dimensions": {
        "training_size": "5k", # "25k", "5k", "1k"
        "training_year": "2012_14", # "2014", "2012_14", "2010_14"
        "scale": "scale", # "scale", "do-not-scale",
        "model": "elasticnet", # "logreg", "penalized_logreg", "rf", "gbm", "elasticnet"
        "cutoff": "none",
        "exclude_features": "age", # "none", "nationality", "sex", "nationality-sex", "age"
        "exclude_subgroups": "drop-non-german", # "keep-all", "drop-non-german"
        "eval_fairness_grouping": "majority-minority"
    }
}


In [ ]:
# Auto-reload the custom package
%load_ext autoreload
%autoreload 1

In [ ]:
from multiversum import Universe

universe = Universe(
    settings=settings
)

# Get the parsed universe settings
dimensions = universe.dimensions
seed = universe.seed


In [ ]:
import numpy as np
parsed_seed = int(seed)
np.random.seed(parsed_seed)
print(f"Using Seed: {parsed_seed}")

# Loading Data

In [ ]:
from pathlib import Path
import pandas as pd

# File paths
raw_file = Path("data/raw/siab.csv")
cache_file = Path("data/00a_siab_multi.csv.gz")

# Ensure cache directory exists
cache_file.parent.mkdir(parents=True, exist_ok=True)

# Load with simple caching
if cache_file.exists():
    print(f"Loading SIAB data from cache: {cache_file}")
    siab = pd.read_csv(cache_file, compression='gzip')
else:
    print(f"Cache not found. Reading raw SIAB data: {raw_file}")
    siab = pd.read_csv(raw_file)
    siab.to_csv(cache_file, index=False, compression='gzip')
    print(f"Cached SIAB data to: {cache_file}")

print(siab.shape)

In [ ]:
siab

# Splitting Data and Setting Training Data Size

In [ ]:
import sys
sys.path.append('..')
from fairness_multiverse.universe import sample_by_year_size

siab_train = sample_by_year_size(
    siab,
    training_year=dimensions["training_year"],
    training_size=dimensions["training_size"],
    random_state=parsed_seed
)


In [ ]:
siab_train.shape

In [ ]:
display(siab_train.groupby("year").size())

In [ ]:
#siab_train = siab_s[siab_s.year < 2015]
siab_calib = siab[siab.year == 2015]
siab_test = siab[siab.year == 2016]

In [ ]:
#siab_calib.shape

In [ ]:
#siab_test.shape

In [ ]:
X_train = siab_train.iloc[:,4:164]
y_train = siab_train.iloc[:, [3]]

In [ ]:
X_train.shape

In [ ]:
X_calib = siab_calib.iloc[:,4:164]
y_calib = siab_calib.iloc[:, [3]]

In [ ]:
X_test = siab_test.iloc[:,4:164]
y_true = siab_test.iloc[:, [3]]

In [ ]:
# Auxiliary data needed downstream in the pipeline

org_train = X_train.copy()
org_test = X_test.copy()
org_calib = X_calib.copy()

# Preprocessing Data

In [ ]:
# EXCLUDE PROTECTED FEATURES
# --------------------------

excluded_features = dimensions["exclude_features"].split("-")
excluded_features_dictionary = {
    "nationality": ["maxdeutsch1", "maxdeutsch.Missing."],
    "sex": ["frau1"],
    "age": ["age"],
}

In [ ]:
excluded_features_columns = [
    excluded_features_dictionary[f] for f in excluded_features if len(f) > 0 and f != "none"
]

In [ ]:
from fairness_multiverse.universe import flatten_once

excluded_features_columns = flatten_once(excluded_features_columns)

In [ ]:
if len(excluded_features_columns) > 0:
    print(f"Dropping features: {excluded_features_columns}")
    X_train.drop(excluded_features_columns, axis=1, inplace=True)

In [ ]:
if len(excluded_features_columns) > 0:
    print(f"Dropping features: {excluded_features_columns}")
    X_test.drop(excluded_features_columns, axis=1, inplace=True)

In [ ]:
if len(excluded_features_columns) > 0:
    print(f"Dropping features: {excluded_features_columns}")
    X_calib.drop(excluded_features_columns, axis=1, inplace=True)

In [ ]:
# EXCLUDE CERTAIN SUBGROUPS
# -------------------------

mode = dimensions.get("exclude_subgroups", "keep-all") # Defaults to "keep-all" if the key is missing.

In [ ]:
if mode == "keep-all":
    keep_mask = pd.Series(True, index=org_train.index)

elif mode == "drop-non-german":
    keep_mask = (org_train["maxdeutsch1"] == 1) & (org_train["maxdeutsch.Missing."] == 0)

else:
    raise ValueError(f"Unsupported mode for exclude_subgroups: {mode}")

In [ ]:
n_drop = (~keep_mask).sum() # Calculates how many rows are set to be dropped
if n_drop > 0:
    pct = n_drop / len(keep_mask) * 100
    print(f"Dropping {n_drop} rows ({pct:.2f}%) where mode='{mode}'")

In [ ]:
X_train = X_train[keep_mask]

In [ ]:
y_train = y_train[keep_mask]

# Model Training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

if (dimensions["model"] == "logreg"):
    model = LogisticRegression() #penalty="none", solver="newton-cg", max_iter=1)
elif (dimensions["model"] == "penalized_logreg"):
    model = LogisticRegression(penalty="l2", C=0.1) #, solver="newton-cg", max_iter=1)
elif (dimensions["model"] == "rf"):
    model = RandomForestClassifier() # n_estimators=100, n_jobs=-1
elif (dimensions["model"] == "gbm"):
    model = GradientBoostingClassifier()
elif (dimensions["model"] == "elasticnet"):
    model = LogisticRegression(penalty = 'elasticnet', solver = 'saga', l1_ratio = 0.5) # max_iter=5000
else:
    raise "Unsupported dimensions.model"

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

model = Pipeline([
    #("continuous_processor", continuous_processor),
    #("categorical_preprocessor", categorical_preprocessor),
    ("scale", StandardScaler() if dimensions["scale"] == "scale" else None),
    ("model", model),
])

In [ ]:
model.fit(X_train, y_train.values.ravel())

In [ ]:
probs_test = model.predict_proba(X_test)
probs_test


In [ ]:
y_pred = pd.Series(model.predict(X_test))
y_pred

# Conformal Prediction

In [ ]:
# Miscoverage level for conformal prediction (10% allowed error rate => 90% target coverage)
alpha = 0.1

In [ ]:
probs_calib = model.predict_proba(X_calib)

In [ ]:
probs_calib

In [ ]:
y_calib = y_calib.values.ravel().astype(int)
y_calib

In [ ]:
from fairness_multiverse.conformal import compute_nc_scores

# Compute nonconformity scores on calibration set (1 - probability of true class)
nc_scores = compute_nc_scores(probs_calib, y_calib)

In [ ]:
from fairness_multiverse.conformal import find_threshold

# Find conformal threshold q_hat for the given alpha (split conformal method)
q_hat = find_threshold(nc_scores, alpha)

In [ ]:
q_hat

In [ ]:
from fairness_multiverse.conformal import predict_conformal_sets

# Generate prediction sets for each test example
pred_sets = predict_conformal_sets(model, X_test, q_hat)

In [ ]:
y_true = y_true.squeeze()

In [ ]:
from fairness_multiverse.conformal import evaluate_sets

# Evaluate coverage and average set size on test data
metrics = evaluate_sets(pred_sets, y_true)

# CP Metrics

In [ ]:
metrics

In [ ]:
example_universe = dimensions.copy()
universe_training_year = example_universe.get("training_year")
universe_training_size = example_universe.get("training_size")
universe_scale = example_universe.get("scale")
universe_model = example_universe.get("model")
universe_exclude_features = example_universe.get("exclude_features")
universe_exclude_subgroups = example_universe.get("exclude_subgroups")

In [ ]:
cp_metrics_dict = {
    "universe_id": [universe.universe_id],
    "universe_training_year": [universe_training_year],
    "universe_training_size": [universe_training_size],
    "universe_scale": [universe_scale],
    "universe_model": [universe_model],
    "universe_exclude_features": [universe_exclude_features],
    "universe_exclude_subgroups": [universe_exclude_subgroups],
    "q_hat": [q_hat],
    "coverage": [metrics["coverage"]],
    "avg_size": [metrics["avg_size"]],
}

In [ ]:
cp_metrics_df = pd.DataFrame(cp_metrics_dict)

In [ ]:
cp_metrics_df

Conditional coverage & looking at subgroups

In [ ]:
from fairness_multiverse.conformal import build_cp_groups

cp_groups_df = build_cp_groups(pred_sets, y_true, X_test.index, org_test)

In [ ]:
# Define covered = 1 if true_label is in the predicted set
cp_groups_df['covered'] = cp_groups_df.apply(
    lambda r: int(r['true_label'] in r['pred_set']),
    axis=1
)

In [ ]:
subgroups = ['frau1','nongerman','nongerman_male','nongerman_female']

# Conditional coverage for subgroup==1
cond_coverage = {
    g: cp_groups_df.loc[cp_groups_df[g]==1, 'covered'].mean()
    for g in subgroups
}
cond_coverage

In [ ]:
for subgroup, cov in cond_coverage.items():
    cp_metrics_df[f"cov_{subgroup}"] = cov

(Non-Ambiguous) Predictions per Subgroup

Ambiguous = Set Size > 1

In [ ]:
cp_groups_df['ambiguous'] = cp_groups_df.apply(
    lambda r: len(r['pred_set']) > 1,
    axis=1
)

In [ ]:
# Conditional ambiguity for subgroup==1 (using same subgroups as above)
cond_ambiguity = {
    g: cp_groups_df.loc[cp_groups_df[g]==1, 'ambiguous'].mean()
    for g in subgroups
}
cond_ambiguity

In [ ]:
for subgroup, amb in cond_ambiguity.items():
    cp_metrics_df[f"frac_ambiguous_{subgroup}"] = amb

In [ ]:
# Compute overall fraction of ambiguous cases
overall_ambiguity = cp_groups_df['ambiguous'].mean()
cp_metrics_df["fraction_ambiguous"] = overall_ambiguity
overall_ambiguity

In [ ]:
cp_metrics_df

In [ ]:

cp_dir = universe.output_dir / "runs" / str(universe.run_no) / "cp_metrics"
cp_dir.mkdir(parents=True, exist_ok=True)
cp_filename = f"cp_{universe.run_no}_{universe.universe_id}.csv"
cp_metrics_df.to_csv(cp_dir / cp_filename, index=False)


# (Fairness) Metrics

In [ ]:
colname_to_bin = "maxdeutsch1"
majority_value = org_train[colname_to_bin].mode()[0]

org_test["majmin"] = np.where(org_test[colname_to_bin] == majority_value, "majority", "minority")

In [ ]:
from fairlearn.metrics import MetricFrame
from typing import Dict, List, Optional, Any, Tuple
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    balanced_accuracy_score,
    f1_score
)
from fairlearn.metrics import (
    false_positive_rate,
    false_negative_rate,
    selection_rate,
    count
)
from fairlearn.metrics import (
    equalized_odds_difference,
    equalized_odds_ratio,
    demographic_parity_difference,
    demographic_parity_ratio,

)

class MetricsCfg:
    metrics = {
        "accuracy": accuracy_score,
        "balanced accuracy": balanced_accuracy_score,
        "f1": f1_score,
        "precision": precision_score,
        "false positive rate": false_positive_rate,
        "false negative rate": false_negative_rate,
        "selection rate": selection_rate,
        "count": count,
    }

    fairness_metrics = {
        "equalized_odds_difference": equalized_odds_difference,
        "equalized_odds_ratio": equalized_odds_ratio,
        "demographic_parity_difference": demographic_parity_difference,
        "demographic_parity_ratio": demographic_parity_ratio,
    }

def compute_metrics(
    sub_universe: Dict,
    y_pred: pd.Series,
    y_test: pd.Series,
    org_test: pd.DataFrame,
) -> Tuple[dict, MetricFrame]:
    """
    Computes a set of metrics for a given sub-universe.

    Args:
        sub_universe: A dictionary containing the parameters for the
            sub-universe.
        y_pred_prob: A pandas series containing the predicted
            probabilities.
        y_test: A pandas series containing the true labels.
        org_test: A pandas dataframe containing the test data, including
            variables that were not used as features.

    Returns:
        A tuple containing two dicst: explicit fairness metrics and
            performance metrics split by fairness groups.
    """
    # Determine cutoff for predictions

    fairness_grouping = sub_universe["eval_fairness_grouping"]
    if fairness_grouping == "majority-minority":
        fairness_group_column = "majmin"
    elif fairness_grouping == "nationality-all":
        fairness_group_column = "maxdeutsch1"
    else:
        raise Exception("invalid value of eval_fairness_grouping")

    assert sub_universe["cutoff"] == "none"

    # Compute fairness metrics (if 2 classes in y_test/y_pred)
    if len(y_test.unique()) != len(y_pred.unique()):
        print(
            "Warning: y_test and y_pred do not have the same number of unique classes."
        )
    if len(y_test.unique()) == 2 and len(y_pred.unique()) == 2:
        fairness_dict = {
                name: metric(
                    y_true=y_test,
                    y_pred=y_pred,
                    sensitive_features=org_test[fairness_group_column],
            )
            for name, metric in MetricsCfg.fairness_metrics.items()
        }
        # sample_params = None
        metrics=MetricsCfg.metrics
    else:
        print(
            "Skipping fairness metrics computation since y_test and y_pred do not have 2 unique classes."
        )
        fairness_dict = {name: None for name in MetricsCfg.fairness_metrics.keys()}

        metrics={ k: v for k, v in MetricsCfg.metrics.items() if k in ["accuracy", "balanced accuracy", "count"]}


    # Compute "normal" metrics (but split by fairness column)
    metric_frame = MetricFrame(
        metrics=metrics,
        y_true=y_test,
        y_pred=y_pred,
        sensitive_features=org_test[fairness_group_column],
        # sample_params=sample_params,
    )

    return (fairness_dict, metric_frame)


In [ ]:
dimensions

In [ ]:
# NOTE: MetricFrame only supports binary scenarios

fairness_dict, metric_frame = compute_metrics(
    sub_universe=dimensions,
    y_test=y_true,
    org_test=org_test,
    y_pred=y_pred
)

# Overall

Main fairness target: Equalized Odds. Seems to be a better fit than equal opportunity, since we're not only interested in Y = 1. Seems to be a better fit than demographic parity, since we also care about accuracy, not just equal distribution of preds.

Pick column for computation of fairness metrics

Performance
Overall performance measures, most interesting in relation to the measures split by group below

In [ ]:
metric_frame.overall

By Group

In [ ]:
metric_frame.by_group

# Final Output

In [ ]:
from multiversum.universe import add_dict_to_df, flatten_dict

final_output = pd.DataFrame()

# Add main fairness metrics to final_output
final_output = add_dict_to_df(final_output, fairness_dict, prefix="fair_main_")
final_output = add_dict_to_df(
    final_output, dict(metric_frame.overall), prefix="perf_ovrl_"
)

# Add group metrics to final output
final_output = add_dict_to_df(
    final_output, flatten_dict(metric_frame.by_group), prefix="perf_grp_"
)

In [ ]:
final_output

In [ ]:
universe.save_data(final_output)
